# 02 - Data Preprocessing for XGBoost

## Purpose

This notebook prepares the processed Olist order dataset for XGBoost modeling.

Based on the data-understanding and EDA stage, this notebook will:
- Load the processed dataset.
- Select the features required for modeling.
- Separate the classification and regression targets.
- Exclude identifier and outcome/leakage columns.
- Handle missing values.
- Transform date columns into usable model features.
- Encode categorical features.
- Prepare the final ML-ready dataset for XGBoost training.

## Feature Decisions

### Selected Features
- customer_state
- primary_seller_state
- primary_category
- item_count
- product_count
- seller_count
- total_price
- total_freight_value
- total_order_value
- avg_item_price
- avg_product_weight_g
- avg_product_length_cm
- avg_product_height_cm
- avg_product_width_cm
- avg_shipping_limit_days
- purchase_year
- purchase_month
- purchase_dayofweek
- order_purchase_timestamp
- order_estimated_delivery_date

### Excluded Columns
- `order_id` — identifier, not a useful model feature
- `customer_id` — identifier, not a useful model feature
- `order_delivered_customer_date` — actual outcome information; causes data leakage
- `delivery_delay_days` — regression target
- `delayed` — classification target

### Targets
- Classification target: `delayed`
- Regression target: `delivery_delay_days`

The final processed data from this notebook will be used by the XGBoost training notebook.


In [13]:
#Import the libraries needed for preprocessing
import pandas as pd
import numpy as np

In [14]:
#Load the processed order dataset
df = pd.read_csv("../../data/processed/processed_orders.csv")

In [3]:
#Check that the dataset loaded correctly
print("Dataset shape:", df.shape)
df.head(2)

Dataset shape: (96476, 25)


,order_id,customer_id,customer_state,primary_seller_state,primary_category,item_count,product_count,seller_count,total_price,total_freight_value,...,avg_product_width_cm,avg_shipping_limit_days,purchase_year,purchase_month,purchase_dayofweek,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,delivery_delay_days,delayed
0,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,SP,PR,health_beauty,3,1,1,134.97,8.49,...,16.0,4.455,2016,9,3,2016-09-15 12:16:38,2016-10-04,2016-11-09 07:47:38,36.325,1
1,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,SP,PR,watches_gifts,1,1,1,29.90,15.56,...,16.0,18.280,2016,10,0,2016-10-03 09:44:50,2016-10-27,2016-10-26 14:02:13,-0.415,0


In [19]:
#Define the finalized ML features and prediction targets
feature_cols = [
    "customer_state",
    "primary_seller_state",
    "primary_category",
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "avg_item_price",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "avg_shipping_limit_days",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

classification_target = "delayed"
regression_target = "delivery_delay_days"

#Verify the number of finalized features
print("Number of features:", len(feature_cols))

Number of features: 20


In [20]:
#Create the feature dataset and separate prediction targets
X = df[feature_cols].copy()
y_class = df[classification_target].copy()
y_reg = df[regression_target].copy()

#Check the dimensions of the features and targets
print("Features:", X.shape)
print("Classification target:", y_class.shape)
print("Regression target:", y_reg.shape)

Features: (96476, 20)
Classification target: (96476,)
Regression target: (96476,)


In [21]:
# Convert timestamp features into numerical date features

timestamp_cols = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    X[col] = pd.to_datetime(X[col])

    X[f"{col}_year"] = X[col].dt.year
    X[f"{col}_month"] = X[col].dt.month
    X[f"{col}_day"] = X[col].dt.day
    X[f"{col}_dayofweek"] = X[col].dt.dayofweek

X = X.drop(columns=timestamp_cols)

print("New feature count:", X.shape[1])

New feature count: 26


In [23]:
# Split the data chronologically and build a leakage-safe preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import joblib
import os

# --------------------------------------------------
# 1. Rebuild X and targets from the original dataset
# --------------------------------------------------

feature_cols = [
    "customer_state",
    "primary_seller_state",
    "primary_category",
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "avg_item_price",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "avg_shipping_limit_days",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

X = df[feature_cols].copy()
y_class = df["delayed"].astype(int).copy()
y_reg = df["delivery_delay_days"].astype(float).copy()

# --------------------------------------------------
# 2. Convert timestamp columns into numerical features
# --------------------------------------------------

timestamp_cols = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    X[col] = pd.to_datetime(X[col])

    X[f"{col}_year"] = X[col].dt.year
    X[f"{col}_month"] = X[col].dt.month
    X[f"{col}_day"] = X[col].dt.day
    X[f"{col}_dayofweek"] = X[col].dt.dayofweek

X = X.drop(columns=timestamp_cols)

# --------------------------------------------------
# 3. Sort chronologically
# --------------------------------------------------

sort_index = pd.to_datetime(
    df["order_purchase_timestamp"]
).sort_values().index

X = X.loc[sort_index].reset_index(drop=True)
y_class = y_class.loc[sort_index].reset_index(drop=True)
y_reg = y_reg.loc[sort_index].reset_index(drop=True)

# --------------------------------------------------
# 4. Create chronological train / validation / test sets
# --------------------------------------------------

n = len(X)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_class_train = y_class.iloc[:train_end].copy()
y_class_val = y_class.iloc[train_end:val_end].copy()
y_class_test = y_class.iloc[val_end:].copy()

y_reg_train = y_reg.iloc[:train_end].copy()
y_reg_val = y_reg.iloc[train_end:val_end].copy()
y_reg_test = y_reg.iloc[val_end:].copy()

# --------------------------------------------------
# 5. Identify categorical and numerical columns
# --------------------------------------------------

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numerical_cols))

# --------------------------------------------------
# 6. Build preprocessing pipelines
# --------------------------------------------------

numeric_pipeline = SimpleImputer(
    strategy="median"
)

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numeric_pipeline, numerical_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ]
)

# --------------------------------------------------
# 7. Fit ONLY on training data
# --------------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# --------------------------------------------------
# 8. Convert processed data into DataFrames
# --------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_val_processed = pd.DataFrame(
    X_val_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

# --------------------------------------------------
# 9. Save the preprocessing pipeline
# --------------------------------------------------

os.makedirs("../../ml/models", exist_ok=True)

joblib.dump(
    preprocessor,
    "../../ml/models/preprocessor.joblib"
)

# --------------------------------------------------
# 10. Save ML-ready datasets
# --------------------------------------------------

output_dir = "../../data/processed/ml_ready"

os.makedirs(output_dir, exist_ok=True)

X_train_processed.to_csv(
    f"{output_dir}/X_train.csv",
    index=False
)

X_val_processed.to_csv(
    f"{output_dir}/X_val.csv",
    index=False
)

X_test_processed.to_csv(
    f"{output_dir}/X_test.csv",
    index=False
)

pd.DataFrame({
    "delayed": y_class_train
}).to_csv(
    f"{output_dir}/y_class_train.csv",
    index=False
)

pd.DataFrame({
    "delayed": y_class_val
}).to_csv(
    f"{output_dir}/y_class_val.csv",
    index=False
)

pd.DataFrame({
    "delayed": y_class_test
}).to_csv(
    f"{output_dir}/y_class_test.csv",
    index=False
)

pd.DataFrame({
    "delivery_delay_days": y_reg_train
}).to_csv(
    f"{output_dir}/y_reg_train.csv",
    index=False
)

pd.DataFrame({
    "delivery_delay_days": y_reg_val
}).to_csv(
    f"{output_dir}/y_reg_val.csv",
    index=False
)

pd.DataFrame({
    "delivery_delay_days": y_reg_test
}).to_csv(
    f"{output_dir}/y_reg_test.csv",
    index=False
)

# --------------------------------------------------
# 11. Final verification
# --------------------------------------------------

print("\nPreprocessing completed successfully!")
print("--------------------------------------")
print("Training set:", X_train_processed.shape)
print("Validation set:", X_val_processed.shape)
print("Test set:", X_test_processed.shape)
print("Final encoded features:", X_train_processed.shape[1])

print("\nSaved:")
print("- X_train.csv")
print("- X_val.csv")
print("- X_test.csv")
print("- y_class_train.csv")
print("- y_class_val.csv")
print("- y_class_test.csv")
print("- y_reg_train.csv")
print("- y_reg_val.csv")
print("- y_reg_test.csv")
print("- preprocessor.joblib")

Categorical features: 3
Numerical features: 23

Preprocessing completed successfully!
--------------------------------------
Training set: (67533, 144)
Validation set: (14471, 144)
Test set: (14472, 144)
Final encoded features: 144

Saved:
- X_train.csv
- X_val.csv
- X_test.csv
- y_class_train.csv
- y_class_val.csv
- y_class_test.csv
- y_reg_train.csv
- y_reg_val.csv
- y_reg_test.csv
- preprocessor.joblib
